In [3]:
import sys
import os
import pandas as pd
import numpy as np
import time
import tracemalloc
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc


import pathlib
repo_root = pathlib.Path().resolve().parent.parent
sys.path.append(str(repo_root))
from shared.preprocessing.preprocess import load_data                           # shared preprocessing — same as rest of group
from shared.evaluation.metrics import evaluate_model, print_evaluation_results      # shared evaluation — keeps results consistent

# ── load and prepare data using shared preprocessing ─────────────────────────
file_path = "../../data/raw/ObesityDataSet_raw_and_data_sinthetic.csv"

print("Loading and preprocessing data...")
data = load_data(file_path)
print(data)

Loading and preprocessing data...
      Gender        Age    Height      Weight family_history_with_overweight  \
0     Female  21.000000  1.620000   64.000000                            yes   
1     Female  21.000000  1.520000   56.000000                            yes   
2       Male  23.000000  1.800000   77.000000                            yes   
3       Male  27.000000  1.800000   87.000000                             no   
4       Male  22.000000  1.780000   89.800000                             no   
...      ...        ...       ...         ...                            ...   
2106  Female  20.976842  1.710730  131.408528                            yes   
2107  Female  21.982942  1.748584  133.742943                            yes   
2108  Female  22.524036  1.752206  133.689352                            yes   
2109  Female  24.361936  1.739450  133.346641                            yes   
2110  Female  23.664709  1.738836  133.472641                            yes   

     

In [ ]:
# Encode categorical features and split into train/test sets
X = data.drop(columns=["NObeyesdad"])
y = data["NObeyesdad"]

# Encode categorical columns in X
for col in X.select_dtypes(include=["object", "string"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Encode target column
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

feature_names = X.columns.tolist()

print("Training samples:", X_train.shape[0])
print("Test samples:    ", X_test.shape[0])
print("Classes:", target_encoder.classes_)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=0)
rf_model.fit(X_train, y_train)

# Output the accuracy on the training and test sets
print("Training set accuracy: {:.3f}".format(rf_model.score(X_train, y_train)))
print("Test set accuracy:     {:.3f}".format(rf_model.score(X_test, y_test)))

# Plot the top feature importances
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]
top_features = [feature_names[i] for i in indices]
top_values = importances[indices]

plt.figure(figsize=(9, 6))
plt.barh(top_features[::-1], top_values[::-1], color="steelblue")
plt.xlabel("Feature Importance Score")
plt.title("Top Feature Importances - Random Forest")
plt.tight_layout()
plt.show()